# 02 — Universal Schema Mapping
**RetailMind · Data Science Practicum 2 · Sai Teja Sunku**

The schema mapper is the **single most important module** in the project —
it's what makes the pipeline universal. Every downstream module reads only
*canonical column names* (`date`, `entity_id`, `sales`, ...), so the same
code runs unchanged on any retail dataset.

| Concept | Module |
|---|---|
| Canonical column roles | `retailmind.schema.ColumnRole` |
| Auto-detection | `retailmind.mapper.SchemaMapper` |
| Smart inference (synthesise revenue, geo→entity, etc.) | `retailmind.smart_inference` |


In [1]:
# Common setup: make the project package importable from the notebooks/ folder
import sys, warnings, json
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px


## 1. The canonical schema
The set of standardised roles every dataset gets mapped to.

In [2]:
from retailmind.schema import ColumnRole

for role in ColumnRole:
    print(f'  {role.value}')

  date
  sales
  entity_id
  quantity
  unit_price
  discount
  profit
  customers
  promo
  holiday
  is_open
  product_id
  product_category
  customer_id
  customer_segment
  region
  city
  state
  ignore
  aux


## 2. Auto-detect on Rossmann

In [3]:
from retailmind.ingest import load_dataset
from retailmind.mapper import SchemaMapper

ros = load_dataset('../train.csv', auxiliary_paths=['../store.csv'])
result = SchemaMapper().infer(ros)
print(result.report())

Canonical schema mapping:
  [✓] date                 → Date
  [✓] sales                → Sales
  [✓] entity_id            → Store
  [ ] quantity             → (none)
  [ ] unit_price           → (none)
  [ ] discount             → (none)
  [ ] profit               → (none)
  [✓] customers            → Customers
  [✓] promo                → Promo
  [✓] holiday              → StateHoliday
  [✓] is_open              → Open
  [ ] product_id           → (none)
  [ ] product_category     → (none)
  [ ] customer_id          → (none)
  [ ] customer_segment     → (none)
  [ ] region               → (none)
  [ ] city                 → (none)
  [ ] state                → (none)
  Auxiliary kept: ['SchoolHoliday', 'StoreType', 'CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear', 'Promo2', 'Promo2SinceWeek', 'Promo2SinceYear', 'PromoInterval', 'DayOfWeek', 'Assortment', 'CompetitionDistance']


## 3. Auto-detect on Walmart
Note how the mapper handles a completely different schema with no `Store` column.

In [4]:
from retailmind.ingest import load_file
wal = load_file('../walmart Retail Data.xlsx')
print(SchemaMapper().infer(wal).report())

Canonical schema mapping:
  [✓] date                 → Order Date
  [✓] sales                → Sales
  [ ] entity_id            → (none)
  [✓] quantity             → Order Quantity
  [✓] unit_price           → Unit Price
  [✓] discount             → Discount
  [✓] profit               → Profit
  [ ] customers            → (none)
  [ ] promo                → (none)
  [ ] holiday              → (none)
  [ ] is_open              → (none)
  [✓] product_id           → Product Name
  [✓] product_category     → Product Category
  [✓] customer_id          → Customer Name
  [✓] customer_segment     → Customer Segment
  [✓] region               → Region
  [✓] city                 → City
  [✓] state                → State
  Auxiliary kept: ['Ship Date', 'Product Base Margin', 'Product Sub-Category', 'Number of Records', 'Customer Age', 'Order ID', 'Order Priority', 'Product Container', 'Row ID', 'Ship Mode', 'Shipping Cost', 'Zip Code']

Warnings:
  ! No entity_id column detected. Pipeline will t

## 4. Smart inference layers
When the dataset has gaps, the smart-inference layer fills them automatically:
- Both `quantity` and `unit_price` exist but no sales → **derive revenue = qty × price**
- No `store` column → **promote `Region` / `Country` to `entity_id`**
- Daily data too sparse → **auto-aggregate to weekly / monthly**

Every decision is logged for transparency.


In [5]:
from retailmind.smart_inference import apply_smart_inference
from retailmind.profiler import profile, DecisionLog
from retailmind.mapper import SchemaMapper

raw_wal = load_file('../walmart Retail Data.xlsx')
pwal = profile(raw_wal)
schema = SchemaMapper().infer(raw_wal).schema
log = DecisionLog()
df, schema, freq = apply_smart_inference(raw_wal, schema, pwal, None, log)
print(f'Chosen aggregation frequency: {freq}')
print()
print(log.render())

Chosen aggregation frequency: D

🔧 **Promoted 'Region' (region) → entity_id** — No store/outlet column was detected, but 'Region' has 4 unique values. Forecasting per-region gives separate models for each, instead of one low-quality global series.


## 5. Manual overrides
Users can override any mapping (used by the Streamlit schema editor).

In [6]:
overrides = {'Region': ColumnRole.ENTITY_ID}
forced = SchemaMapper().infer(raw_wal, overrides=overrides)
print(forced.schema.summary())

Canonical schema mapping:
  [✓] date                 → Order Date
  [✓] sales                → Sales
  [✓] entity_id            → Region
  [✓] quantity             → Order Quantity
  [✓] unit_price           → Unit Price
  [✓] discount             → Discount
  [✓] profit               → Profit
  [ ] customers            → (none)
  [ ] promo                → (none)
  [ ] holiday              → (none)
  [ ] is_open              → (none)
  [✓] product_id           → Product Name
  [✓] product_category     → Product Category
  [✓] customer_id          → Customer Name
  [✓] customer_segment     → Customer Segment
  [ ] region               → (none)
  [✓] city                 → City
  [✓] state                → State
  Auxiliary kept: ['Ship Date', 'Product Base Margin', 'Product Sub-Category', 'Number of Records', 'Customer Age', 'Order ID', 'Order Priority', 'Product Container', 'Row ID', 'Ship Mode', 'Shipping Cost', 'Zip Code']


## Summary
- `SchemaMapper` auto-maps raw columns to canonical roles
- `smart_inference` fills gaps (revenue synthesis, geo fallback, aggregation)
- Manual overrides keep the user in control when auto-detection is wrong

**Next:** [03 — Feature Engineering](03_feature_engineering.ipynb)
